In [2]:
import numpy as np
import matplotlib.pyplot as plt


# ---------------------- 1. 生成模拟的ARMA(2,1)数据 ----------------------
def generate_arma_data(p=2, q=1, n=1000, phi=[0.5, -0.2], theta=[0.3], c=1.0):
    """
    生成ARMA(p,q)模拟数据
    :param p: AR阶数
    :param q: MA阶数
    :param n: 数据长度
    :param phi: AR参数列表
    :param theta: MA参数列表
    :param c: 常数项
    :return: 生成的时间序列y，白噪声epsilon
    """
    # 初始化白噪声（服从正态分布）
    epsilon = np.random.normal(0, 1, n + max(p, q))
    # 初始化时间序列
    y = np.zeros(n + max(p, q))

    # 生成ARMA序列
    for t in range(max(p, q), n + max(p, q)):
        # AR部分
        ar_part = np.sum(phi * y[t - p : t][::-1])
        # MA部分
        ma_part = np.sum(theta * epsilon[t - q : t][::-1])
        # ARMA公式
        y[t] = c + ar_part + epsilon[t] - ma_part

    # 去掉前max(p,q)个初始化值，保证数据纯净
    y = y[max(p, q) :]
    epsilon = epsilon[max(p, q) :]
    return y, epsilon


# 生成模拟数据（已知真实参数：phi=[0.5,-0.2], theta=[0.3], c=1.0）
y, true_epsilon = generate_arma_data(n=1000)


# ---------------------- 2. 拟合AR(p)参数（Yule-Walker方程） ----------------------
def fit_ar_params(y, p):
    """
    用Yule-Walker方程拟合AR(p)参数
    :param y: 时间序列
    :param p: AR阶数
    :return: AR参数phi，常数项c
    """
    # 计算序列的均值（用于去均值）
    y_mean = np.mean(y)
    y_centered = y - y_mean  # 去均值序列

    # 计算自协方差函数（0到p阶）
    gamma = np.zeros(p + 1)
    for k in range(p + 1):
        # 关键修复：k=0时使用y_centered[:]而非y_centered[:-k]
        if k == 0:
            # 0阶自协方差 = 序列方差（所有元素自乘求和）
            gamma[k] = np.sum(y_centered * y_centered) / len(y_centered)
        else:
            # k>0时正常计算滞后k阶的自协方差
            gamma[k] = np.sum(y_centered[k:] * y_centered[:-k]) / len(y_centered)

    # 构建Yule-Walker矩阵
    R = np.zeros((p, p))
    for i in range(p):
        for j in range(p):
            R[i, j] = gamma[abs(i - j)]

    # 构建右侧向量
    b = gamma[1 : p + 1]

    # 求解线性方程组 R*phi = b
    phi = np.linalg.solve(R, b)

    # 计算常数项c（c = 均值 * (1 - sum(phi))）
    c = y_mean * (1 - np.sum(phi))

    return phi, c


# 拟合AR(2)参数
p = 2
fitted_phi, fitted_c = fit_ar_params(y, p)
print(f"真实AR参数: [0.5, -0.2] | 拟合AR参数: {np.round(fitted_phi, 4)}")
print(f"真实常数项: 1.0 | 拟合常数项: {np.round(fitted_c, 4)}")


# ---------------------- 3. 计算AR部分的残差 ----------------------
def calculate_ar_residuals(y, phi, c, p):
    """
    计算AR模型的残差（用于拟合MA参数）
    :param y: 时间序列
    :param phi: AR参数
    :param c: 常数项
    :param p: AR阶数
    :return: 残差序列
    """
    residuals = np.zeros_like(y)
    # 前p个值无法计算残差，用0填充
    for t in range(p, len(y)):
        ar_part = np.sum(phi * y[t - p : t][::-1])
        residuals[t] = y[t] - c - ar_part
    # 去掉前p个无效值
    residuals = residuals[p:]
    return residuals


ar_residuals = calculate_ar_residuals(y, fitted_phi, fitted_c, p)


# ---------------------- 4. 拟合MA(q)参数（最小二乘法） ----------------------
def fit_ma_params(residuals, q):
    """
    用最小二乘法拟合MA(q)参数
    :param residuals: AR模型的残差
    :param q: MA阶数
    :return: MA参数theta
    """
    # 构建MA拟合的设计矩阵X
    n = len(residuals)
    X = np.zeros((n - q, q))
    y_ma = residuals[q:]  # 目标变量

    # 填充设计矩阵（每一行是前q个残差）
    for t in range(q, n):
        X[t - q] = residuals[t - q : t][::-1]

    # 最小二乘法求解：theta = (X^T X)^-1 X^T y_ma
    theta = np.linalg.inv(X.T @ X) @ X.T @ y_ma
    return theta


# 拟合MA(1)参数
q = 1
fitted_theta = fit_ma_params(ar_residuals, q)
print(f"真实MA参数: [0.3] | 拟合MA参数: {np.round(fitted_theta, 4)}")


# ---------------------- 5. 验证拟合效果（预测） ----------------------
def arma_predict(y, phi, theta, c, p, q, n_steps=100):
    """
    用拟合的ARMA参数进行预测
    :param y: 原始序列
    :param phi: AR参数
    :param theta: MA参数
    :param c: 常数项
    :param p: AR阶数
    :param q: MA阶数
    :param n_steps: 预测步数
    :return: 预测值
    """
    # 初始化预测结果和残差
    y_pred = np.zeros(len(y) + n_steps)
    y_pred[: len(y)] = y
    residuals = calculate_ar_residuals(y, phi, c, p)
    residuals = np.pad(residuals, (0, n_steps), mode="constant")  # 补0

    # 逐步预测
    for t in range(len(y), len(y) + n_steps):
        # AR部分
        ar_part = np.sum(phi * y_pred[t - p : t][::-1])
        # MA部分
        ma_part = np.sum(theta * residuals[t - q : t][::-1])
        # 预测值
        y_pred[t] = c + ar_part + ma_part
        # 更新残差（预测残差 = 预测值 - 实际值，这里实际值未知，用0近似）
        residuals[t] = y_pred[t] - (c + ar_part)

    return y_pred


# 预测未来100步
y_pred = arma_predict(y, fitted_phi, fitted_theta, fitted_c, p, q, n_steps=100)

# 可视化结果
plt.figure(figsize=(12, 6))
plt.plot(range(len(y)), y, label="原始数据", color="blue")
plt.plot(
    range(len(y), len(y) + 100),
    y_pred[len(y) :],
    label="ARMA预测",
    color="red",
    linestyle="--",
)
plt.title("ARMA(2,1)模型拟合与预测结果")
plt.xlabel("时间步")
plt.ylabel("值")
plt.legend()
plt.grid(True)
plt.show()

真实AR参数: [0.5, -0.2] | 拟合AR参数: [ 0.2356 -0.1261]
真实常数项: 1.0 | 拟合常数项: 1.2675
真实MA参数: [0.3] | 拟合MA参数: [-0.0037]


IndexError: index 1098 is out of bounds for axis 0 with size 1098